# Scopus-Based Text Mining Mini-Project (Template)

This Jupyter Notebook is a **template** for your integrated text mining / NLP project using **Scopus** data.

You will:

1. Define a **research question** based on a theme (e.g., *climate justice*, *urban circularity*, *migration integration*, *gender bias in AI*).
2. Use **Scopus** to collect a corpus of scientific articles on that theme.
3. Build a **clean text corpus** (titles, abstracts, keywords).
4. Apply a **preprocessing pipeline**:
   - cleaning  
   - tokenization  
   - stopword removal  
   - lemmatization  
   - n-grams  
5. Transform text into **numerical representations** (Bag-of-Words, TF-IDF).
6. Conduct **exploratory analyses** (word frequencies, n-grams, trends over time).
7. Optionally, apply a **simple model**, such as:
   - clustering (e.g. KMeans)  
   - topic modeling (e.g. LDA)  
8. Interpret results from a **social science / humanities perspective**.

> ⚠️ This notebook is a **template**. You must **adapt the code and comments** to your specific topic and dataset.


## 1. Define Your Research Question

In this section, describe **in your own words**:

- Your **theme** (e.g., *urban circularity*).  
- Your **research question** (e.g., *How has the concept of urban circularity evolved in the scientific literature since 2000?*).  
- The **type of insights** you aim to obtain using text mining (topics, discourse shifts, key concepts, etc.).

**Example** (replace with your own):
- *Theme*: Climate justice  
- *Research question*: *How has the framing of climate justice changed over time across disciplines in the scientific literature?*


➡️ **Your research question (edit below):**

> *Theme*:  
> *Research question*:  
> *Expected type of insights*:  


## 2. Scopus Data Collection (Outside Python)

You should now:

1. Access **Scopus** via your institution (e.g., through your library portal).  
2. Build a **search query** using keywords, Boolean operators, and filters (years, subject areas, document types).  
3. Export the results as a **CSV file**, including at least:  
   - `Title`  
   - `Abstract`  
   - `Author Keywords` (or `Index Keywords` if available)  
   - `Year`  
   - `Authors`  
   - `Source title` (Journal / Conference)

Suggested file name (you can adjust):  
- `data/scopus_export.csv`

Once you have the file exported and saved locally (in the same project folder as this notebook), you can proceed.


## 3. Setup: Import Libraries

In [ ]:
# Standard libraries
import os
import pandas as pd
import numpy as np

# Text processing
import re
import spacy

# Vectorization
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Optional: modeling / clustering
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Progress bar
from tqdm.auto import tqdm

# Display options
pd.set_option("display.max_colwidth", 200)

print("Libraries imported.")


### 3.1 Load spaCy Model

Make sure you installed the English model once in your environment (in a terminal):

```bash
python -m spacy download en_core_web_sm
```

Then run the cell below.


In [ ]:
# Load spaCy English model
nlp = spacy.load("en_core_web_sm")
nlp


## 4. Load Scopus Data

Adjust the file path below to the location of your **Scopus CSV export**.


In [ ]:
# Path to your Scopus CSV export
data_path = "data/scopus_export.csv"  # adjust if needed

if not os.path.exists(data_path):
    print(f"⚠️ File not found at: {data_path}")
    print("Please adjust `data_path` to point to your Scopus CSV.")
else:
    df = pd.read_csv(data_path)
    print("Data loaded. Shape:", df.shape)
    display(df.head())


### 4.1 Build a Combined Text Field

We will typically combine:
- **Title**
- **Abstract**
- **Author Keywords** (if available)

into a single free-text field for analysis.


In [ ]:
# Adjust column names depending on your Scopus export
possible_title_cols = ["Title", "Document Title", "title"]
possible_abstract_cols = ["Abstract", "Abstracts", "abstract"]
possible_keyword_cols = ["Author Keywords", "Authors Keywords", "authkeywords", "authkeywords_full"]

def find_first_existing(colnames):
    for c in colnames:
        if c in df.columns:
            return c
    return None

title_col = find_first_existing(possible_title_cols)
abstract_col = find_first_existing(possible_abstract_cols)
keywords_col = find_first_existing(possible_keyword_cols)

print("Using columns:")
print("  Title   :", title_col)
print("  Abstract:", abstract_col)
print("  Keywords:", keywords_col)

def safe_text(x):
    if isinstance(x, str):
        return x
    elif pd.isna(x):
        return ""
    else:
        return str(x)

df["text_raw"] = (
    df[title_col].apply(safe_text) + " " +
    df[abstract_col].apply(safe_text) + " " +
    (df[keywords_col].apply(safe_text) if keywords_col else "")
)

df["text_raw"] = df["text_raw"].fillna("").astype(str)

df[["text_raw"]].head()


## 5. Text Preprocessing Pipeline

We will implement the following steps:

1. Cleaning (lowercase, remove digits and extra spaces).  
2. Tokenization with spaCy.  
3. Stopword removal.  
4. Lemmatization.  
5. (Optional) custom stopword removal.  
6. Join tokens back into processed text.

This corresponds to the workflow you saw in the Markdown guide.


In [ ]:
# Basic cleaning function using regex
def basic_clean(text: str) -> str:
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # Remove digits (optional)
    text = re.sub(r"\d+", " ", text)
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Custom domain-specific stopwords (edit for your project)
custom_stopwords = set([
    "study", "paper", "result", "analysis", "article",
    # add domain-specific frequent terms here if needed
])

def preprocess_text(text: str, nlp_model, remove_custom=True) -> str:
    """
    Full preprocessing pipeline for a single document:
    - basic clean
    - tokenize with spaCy
    - remove stopwords & non-alphabetic tokens
    - lemmatize
    - apply custom stopwords (optional)
    """
    text = basic_clean(text)
    doc = nlp_model(text)
    tokens = []
    for token in doc:
        # keep alphabetic tokens only
        if not token.is_alpha:
            continue
        # spaCy stopwords
        if token.is_stop:
            continue
        lemma = token.lemma_.strip()
        if not lemma:
            continue
        # custom stopwords
        if remove_custom and lemma in custom_stopwords:
            continue
        tokens.append(lemma)
    return " ".join(tokens)

# Test the preprocessing on a single example
if 'df' in globals() and not df.empty:
    example_text = df["text_raw"].iloc[0]
else:
    example_text = "Urban circularity aims to optimize resource flows in cities."

print("RAW TEXT:\n", example_text[:500])
print("\nPREPROCESSED:\n", preprocess_text(example_text, nlp))


### 5.1 Apply Preprocessing to the Full Corpus

This may take some time depending on the size of your dataset.


In [ ]:
if 'df' in globals():
    tqdm.pandas()
    df["text_clean"] = df["text_raw"].progress_apply(lambda x: preprocess_text(x, nlp))
    df[["text_raw", "text_clean"]].head()
else:
    print("⚠️ Dataframe `df` not loaded. Please fix the data path and rerun.")


## 6. Vectorization: Bag-of-Words & TF-IDF (Unigrams + Bigrams)

We will now convert the preprocessed text into numerical vectors using:

- **Bag-of-Words (BoW)** — simple counts of tokens  
- **TF-IDF** — scaled counts that emphasize informative terms  

We will include:
- unigrams (1-grams)  
- bigrams (2-grams)

to capture multi-word concepts like “climate change”, “social justice”, “energy efficiency”.


In [ ]:
if 'df' in globals():
    texts = df["text_clean"].fillna("").tolist()
    print("Number of documents:", len(texts))
else:
    texts = []
    print("⚠️ No texts loaded.")


### 6.1 Bag-of-Words (Counts)

In [ ]:
# Bag-of-Words with unigrams + bigrams
bow_vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=5)  # min_df=5 to reduce noise
if texts:
    X_bow = bow_vectorizer.fit_transform(texts)
    print("BoW matrix shape:", X_bow.shape)  # (n_docs, vocab_size)
else:
    X_bow = None
    print("⚠️ Empty texts list; cannot build BoW matrix.")


### 6.2 TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=5)
if texts:
    X_tfidf = tfidf_vectorizer.fit_transform(texts)
    print("TF-IDF matrix shape:", X_tfidf.shape)
else:
    X_tfidf = None
    print("⚠️ Empty texts list; cannot build TF-IDF matrix.")


## 7. Exploratory Text Analysis

We will now examine:

- Most frequent terms (BoW)  
- Highest-weighted TF-IDF terms  


In [ ]:
# Helper: get top terms from BoW
def get_top_terms_bow(vectorizer, X, top_n=20):
    sums = np.array(X.sum(axis=0)).flatten()
    terms = vectorizer.get_feature_names_out()
    data = pd.DataFrame({"term": terms, "count": sums})
    return data.sort_values("count", ascending=False).head(top_n)

if 'X_bow' in globals() and X_bow is not None:
    top_terms_bow = get_top_terms_bow(bow_vectorizer, X_bow, top_n=30)
    display(top_terms_bow)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_terms_bow, x="count", y="term")
    plt.title("Top Terms (BoW counts)")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ BoW matrix not available.")


In [ ]:
# Helper: approximate "global" TF-IDF importance by max across documents
def get_top_terms_tfidf(vectorizer, X, top_n=20):
    # Take max TF-IDF score for each term across all documents
    max_vals = X.max(axis=0).toarray().flatten()
    terms = vectorizer.get_feature_names_out()
    data = pd.DataFrame({"term": terms, "max_tfidf": max_vals})
    return data.sort_values("max_tfidf", ascending=False).head(top_n)

if 'X_tfidf' in globals() and X_tfidf is not None:
    top_terms_tfidf = get_top_terms_tfidf(tfidf_vectorizer, X_tfidf, top_n=30)
    display(top_terms_tfidf)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_terms_tfidf, x="max_tfidf", y="term")
    plt.title("Top Terms (TF-IDF, max across docs)")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ TF-IDF matrix not available.")


## 8. Optional: Topic Modeling with LDA (Latent Dirichlet Allocation)

We can use **LDA** on the Bag-of-Words matrix to discover latent topics.

> ⚠️ Adjust `n_topics` based on your corpus size and research question.


In [ ]:
n_topics = 5  # adjust

if 'X_bow' in globals() and X_bow is not None:
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method="batch"
    )
    lda_topics = lda.fit_transform(X_bow)
    print("LDA fitted. Topic distribution shape:", lda_topics.shape)
else:
    lda = None
    print("⚠️ X_bow not available; cannot fit LDA.")


In [ ]:
def print_lda_topics(model, vectorizer, top_n=15):
    terms = vectorizer.get_feature_names_out()
    for idx, topic in enumerate(model.components_):
        print(f"\n### Topic {idx}")
        top_indices = topic.argsort()[::-1][:top_n]
        topic_terms = [terms[i] for i in top_indices]
        print(", ".join(topic_terms))

if 'lda' in globals() and lda is not None:
    print_lda_topics(lda, bow_vectorizer, top_n=15)
else:
    print("⚠️ LDA model not available.")


## 9. Interpretation (Social Science / Humanities Perspective)

Use this section to **interpret your results**:

1. **Top terms**:  
   - What concepts dominate the literature?  
   - Do these reflect specific subfields or disciplines?  

2. **N-grams**:  
   - What stable *phrases* appear (e.g., "climate justice", "urban resilience")?  

3. **Topics (if you used LDA)**:  
   - Give each topic a **short label** (e.g., “Governance & policy”, “Technical mitigation”, “Social movements”).  
   - How do these topics relate to your research question?  

4. **Temporal trends** (if you analyze by year):  
   - Are some topics or terms emerging or declining over time?  
   - Are new concepts appearing in recent years?  

5. **Critical reflection**:  
   - What are the **limitations** of your corpus (e.g., language bias, disciplinary bias)?  
   - What are the **limitations** of the methods (e.g., bag-of-words ignoring context)?  
   - How could more advanced methods (embeddings, transformers) improve the analysis?  


➡️ **Write your interpretation here (edit this cell):**

> **Summary of key findings:**  
>
> **How do they answer (or relate to) your research question?**  
>
> **What are the main limitations of your analysis?**  
>
> **What would be a natural next step (more data, better models, qualitative follow-up, etc.)?**
